# Fall 2024 Data Science Track: Week 2 - Data Cleaning Exercise

## Packages, Packages, Packages!

Import *all* the things here! You need the standard stuff: `pandas` and `numpy`.

If you got more stuff you want to use, add them here too. 🙂

In [ ]:
import pandas as pd
import numpy as np


## Introduction

With the packages out of the way, now you will be working with the following data sets:

* `food_coded.csv`: [Food choices](https://www.kaggle.com/datasets/borapajo/food-choices?select=food_coded.csv) from Kaggle
* `Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv`: [Ask A Manager Salary Survey 2021 (Responses)](https://docs.google.com/spreadsheets/d/1IPS5dBSGtwYVbjsfbaMCYIWnOuRmJcbequohNxCyGVw/view?&gid=1625408792) as *Tab Separated Values (.tsv)* from Google Docs

Each one poses different challenges. But you’ll―of course―overcome them with what you learned in class! 😉

## Food Choices Data Set

### Load the Data

In [ ]:
# Load the Food choices data set into a variable (e.g., df_food).

food_data_set_path = 'data/food_coded.csv'

df_food = pd.read_csv(food_data_set_path)

### Explore the Data

How much data did you just load?

In [ ]:
len(df_food)


What are the columns and their types in this data set?

In [ ]:
df_food.dtypes


### Clean the Data

Perhaps we’d like to know more another day, but the team is really interested in just the relationship between calories (`calories_day`) and weight. …and maybe gender.

Can you remove the other columns?

In [ ]:
df_food = df_food[['Gender', 'calories_day', 'weight']]
df_food.head()


What about `NaN`s? How many are there?

In [ ]:
df_food.isna().sum()


We gotta remove those `NaN`s―the entire row.

In [ ]:
df_food = df_food.dropna()
df_food.head()


But what about the weird non-numeric values in the column obviously meant for numeric data?

Notice the data type of that column from when you got the types of all the columns?

If only we could convert the column to a numeric type and drop the rows with invalid values. 🤔

In [ ]:
df_food['calories_day'] = pd.to_numeric(df_food['calories_day'], errors='coerce')
df_food = df_food.dropna(subset=['calories_day'])
df_food['weight'] = pd.to_numeric(df_food['weight'], errors='coerce')
df_food = df_food.dropna(subset=['weight'])
df_food.head()


Now this data seems reasonably clean for our purposes! 😁

Let’s save it somewhere to be shipped off to another teammate. 💾

In [ ]:
df_food.to_csv('data/food_cleaned.csv', index=False)
df_food.head()


## Ask a Manager Salary Survey 2021 (Responses) Data Set

### Load the Data

In [ ]:
salary_data_path = 'data/Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv'
df_salary = pd.read_csv(salary_data_path, sep='\t')
df_salary.head()


Was that hard? 🙃

#### rename the file to something that is better for all systems.  
* No spaces in filename (can use '_')
* all lower case

### Explore

You know the drill.

How much data did you just load?

In [ ]:
len(df_salary)


What are the columns and their types?

In [ ]:
df_salary.dtypes


Oh… Ugh! Give these columns easier names to work with first. 🙄

In [ ]:
df_salary = df_salary.rename(columns={
    'Timestamp': 'timestamp',
    'How old are you?': 'age',
    'What industry do you work in?': 'industry',
    'Job title': 'title',
    'If your job title needs additional context, please clarify here:': 'title_context',
    'What is your annual salary? (You\'ll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)': 'salary',
    'How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.': 'additional_compensation',
    'Please indicate the currency': 'currency',
    'If "Other," please indicate the currency here: ': 'other_currency',
    'If your income needs additional context, please provide it here:': 'salary_context',
    'What country do you work in?': 'country',
    "If you're in the U.S., what state do you work in?": 'state',
    'What city do you work in?': 'city',
    'How many years of professional work experience do you have overall?': 'total_yoe',
    'How many years of professional work experience do you have in your field?': 'field_yoe',
    'What is your highest level of education completed?': 'highest_education_completed',
    'What is your gender?': 'gender',
    'What is your race? (Choose all that apply.)': 'race'
})

df_salary.head()


It’s a lot, and that should not have been easy. 😏

You’re going to have a gander at the computing/tech subset first because thats *your* industry. But first, what value corresponds to that `industry`?

In [ ]:
df_salary['industry'].value_counts()


That value among the top 5 is what you’re looking for innit? Filter out all the rows not in that industry and save it into a new dataframe. 

In [ ]:
df_salary_tech = df_salary[df_salary['industry'] == 'Computing or Tech'].copy()
df_salary_tech.head()


Do a sanity check to make sure that the only values you kept are the one you are filtered for.  

In [ ]:
df_salary_tech['industry'].unique()


We are very interested in salary figures. But how many dollars 💵 is a euro 💶 or a pound 💷? That sounds like a problem for another day. 🫠

For now, let’s just look at U.S. dollars (`'USD'`).

In [ ]:
df_salary_tech = df_salary_tech[df_salary_tech['currency'] == 'USD'].copy()
df_salary_tech.head()


What we really want know is how each U.S. city pays in tech. What value in `country` represents the United States of America?

In [ ]:
df_salary_tech['country'].value_counts().head(10)


### Clean the Data

Well, we can’t get our answers with what we currently have, so you’ll have to make some changes.

Let’s not worry about anything below the first 5 values for now. Convert the top 5 to a single canonical value―say, `'US'`, which is nice and short.

In [ ]:
country_replacements = {
    'United States': 'US',
    'USA': 'US',
    'US': 'US',
    'U.S.': 'US',
    'United States of America': 'US',
    'United states': 'US',
    'united states': 'US',
    'Usa': 'US',
    'usa': 'US',
    'Us': 'US'
}

df_salary_tech['country'] = df_salary_tech['country'].replace(country_replacements)
df_salary_tech['country'].value_counts().head(10)


Have a look at the count of each unique country again now.

In [ ]:
df_salary_tech['country'].value_counts().head(10)


Did you notice anything interesting?

In [ ]:
# BONUS CREDIT: resolve [most of] those anomalous cases too without exhaustively taking every variant literally into account.



In [ ]:

# BONUS CREDIT: if you’ve resolved it, let’s see how well you did by counting the number of instances of each unique value.



It’s looking good so far. Let’s find out the minimum, mean, and maximum (in that order) salary by state, sorted by the mean in descending order.

In [ ]:
df_salary_tech['salary'] = df_salary_tech['salary'].replace('[\$,]', '', regex=True)
df_salary_tech['salary'] = pd.to_numeric(df_salary_tech['salary'], errors='coerce')

df_salary_tech.groupby('state')['salary'].agg(['min', 'mean', 'max']).sort_values('mean', ascending=False).head(10)


Well, pooh! We forgot that `salary` isn’t numeric. Something wrong must be fixed.

In [ ]:
df_salary_tech['salary'] = df_salary_tech['salary'].astype(str)
df_salary_tech['salary'] = df_salary_tech['salary'].str.replace('[\$,]', '', regex=True)
df_salary_tech['salary'] = pd.to_numeric(df_salary_tech['salary'], errors='coerce')
df_salary_tech['salary'].head()


Let’s try that again.

In [ ]:
df_salary_tech.groupby('state')['salary'].agg(['min', 'mean', 'max']).sort_values('mean', ascending=False).head(10)


That did the trick! Now let’s narrow this to data 2021 and 2022 just because (lel). *(Hint: that timestamp column may not be a temporal type right now.)*

In [ ]:
# Filter the data to within 2021, 2022, or 2023, saving the DataFrame to a new variable, and generate the summary again.
df_salary_tech['timestamp'] = pd.to_datetime(df_salary_tech['timestamp'], errors='coerce')
df_salary_tech_2021_2023 = df_salary_tech[df_salary_tech['timestamp'].dt.year.isin([2021, 2022, 2023])].copy()

df_salary_tech_2021_2023.groupby('state')['salary'].agg(['min', 'mean', 'max']).sort_values('mean', ascending=False).head(10)


## Bonus

Clearly, we do not have enough data to produce useful figures for the level of specificity you’ve now reached. What do you notice about Delaware and West Virginia?

Let’s back out a bit and return to `df_salary` (which was the loaded data with renamed columns but *sans* filtering).

### Bonus #0

Apply the same steps as before to `df_salary`, but do not filter for any specific industry. Do perform the other data cleaning stuff, and get to a point where you can generate the minimum, mean, and maximum by state.

### Bonus #1

This time, format the table output nicely (*$12,345.00*) without modifying the values in the `DataFrame`. That is, `df_salary` should be identical before versus after running your code.

(*Hint: if you run into an error about `jinja2` perhaps you need to `pip install` something.*)

### Bonus #2

Filter out the non-single-states (e.g., `'California, Colorado'`) in the most elegant way possible (i.e., *not* by blacklisting all the bad values).

### Bonus #3

Show the quantiles instead of just minimum, mean, and maximum―say 0%, 5%, 25%, 50%, 75%, 95%, and 100%. Outliers may be deceiving.

Sort by whatever interests you―like say the *50th* percentile.

And throw in a count by state too. It would be interesting to know how many data points contribute to the figures for each state. (*Hint: your nice formatting from Bonus #1 might not work this time around.* 😜)